# Macro feature screening and forecast-safe engineering

Notebook này nghiên cứu feature cho target `unemployment_rate`. Mọi phép chọn chỉ dùng train đến `2017-12`; validation và test được khóa. Các engineered signal đều dùng thông tin trễ ít nhất một tháng.

## Phương pháp

1. Loại biến được tạo trực tiếp từ target để tránh leakage.
2. Tạo lag, MoM, YoY, rolling mean/std theo cách forecast-safe.
3. So từng signal với baseline gồm 12 lag của target bằng expanding-window CV trên train.
4. Ghi thêm AIC gain để audit, nhưng quyết định chính dựa trên out-of-sample RMSE gain và độ ổn định qua folds.
5. Mỗi biến gốc giữ tối đa một signal tốt nhất và loại signal tương quan quá cao với signal đã chọn.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'macro_indicators.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PACKAGE_SRC = PROJECT_ROOT / 'vmd_ardl_ffnn' / 'src'
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from vmd_ardl_ffnn.macro_features import (
    FeatureResearchConfig,
    build_eligibility_table,
    engineer_forecast_safe_features,
    load_macro_frame,
    run_feature_screening,
    write_feature_research_artifacts,
)

DATA_PATH = PROJECT_ROOT / 'macro_indicators.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'macro-feature-screening'
config = FeatureResearchConfig()

: 

## 1. Load và audit dữ liệu

In [ ]:
raw = load_macro_frame(DATA_PATH, config)
print(f'Rows: {len(raw):,}')
print(f"Range: {raw[config.date_column].min().date()} -> {raw[config.date_column].max().date()}")
display(raw.head())
display(raw.isna().sum().rename('missing').to_frame().sort_values('missing', ascending=False))

In [ ]:
eligibility = build_eligibility_table(raw, config)
display(eligibility)
print('Leakage-risk columns are explicitly excluded:')
display(eligibility[eligibility['reason'] == 'derived_from_target_leakage_risk'])

In [ ]:
raw.set_index(config.date_column)[config.target].plot(figsize=(12, 4), title='Unemployment rate')
plt.axvline(config.train_end, color='tab:green', linestyle='--', label='train end')
plt.axvline(config.validation_end, color='tab:orange', linestyle='--', label='validation end')
plt.legend()
plt.ylabel(config.target)
plt.show()

## 2. Forecast-safe feature engineering

Tên signal thể hiện rõ transformation và lag. Ví dụ `initial_claims__mom_1_lag_1` là thay đổi MoM đã trễ một tháng; không signal nào sử dụng giá trị exog cùng tháng với target.

In [ ]:
engineered, catalog = engineer_forecast_safe_features(raw, config)
print(f'Engineered signals: {len(catalog):,}')
display(catalog.groupby(['base_feature', 'transformation']).size().rename('count').reset_index())
assert catalog['leakage_safe'].all()
assert catalog['availability_lag'].min() >= 1

## 3. Train-only expanding-window screening

`mean_rmse_gain_vs_target_lags > 0` nghĩa là signal cải thiện so với baseline chỉ dùng lag của thất nghiệp. `improvement_rate` là tỷ lệ folds có cải thiện. Validation và test không tham gia bước này.

In [ ]:
result = run_feature_screening(raw, config)
display(result.split_audit)
display(result.scores.head(25).style.format({
    'mean_candidate_rmse': '{:.4f}',
    'mean_rmse_gain_vs_target_lags': '{:.4f}',
    'mean_mae_gain_vs_target_lags': '{:.4f}',
    'improvement_rate': '{:.0%}',
    'rmse_gain_std': '{:.4f}',
    'aic_gain_vs_target_lags': '{:.2f}',
}))

In [ ]:
top = result.scores.head(20).sort_values('mean_rmse_gain_vs_target_lags')
colors = ['tab:green' if value > 0 else 'tab:red' for value in top['mean_rmse_gain_vs_target_lags']]
ax = top.plot.barh(
    x='signal', y='mean_rmse_gain_vs_target_lags', color=colors,
    figsize=(10, 8), legend=False, title='Rolling-CV RMSE gain vs target-lag baseline'
)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('RMSE gain (positive is better)')
plt.tight_layout()
plt.show()

## 4. Stable, non-redundant recommendation

In [ ]:
print('Selected base features for the VMD–ARDL–FFNN pipeline:')
display(pd.DataFrame({'base_feature': result.selected_base_features, 'best_signal': result.selected_signals}))

if result.selected_signals:
    selected_corr = result.research_matrix.loc[:config.train_end, list(result.selected_signals)].corr()
    display(selected_corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format('{:.2f}'))
else:
    print('No signal met the positive-gain and stability thresholds; keep the target-only baseline.')

## 5. Export audit artifacts

`selected_features.json` chứa biến gốc khuyến nghị để pipeline tiếp tục tự chọn lag theo từng component VMD. Engineered signals được xuất riêng cho nghiên cứu, tránh vô tình lag hai lần trong production pipeline.

In [ ]:
output = write_feature_research_artifacts(result, OUTPUT_DIR)
print(f'Artifacts written to: {output}')
for path in sorted(output.iterdir()):
    print('-', path.name)

## Cách diễn giải

- Rolling-CV gain là tiêu chí chính; AIC chỉ là diagnostic bổ sung.
- Kết quả này chọn feature, chưa mở test để báo cáo hiệu năng cuối.
- Sau khi chốt danh sách, hãy đưa `selected_base_features` vào cấu hình pipeline và để ARDL chọn lag theo từng component.
- Nếu thay đổi feature policy hoặc dữ liệu, chạy lại notebook và lưu artifact mới để đảm bảo truy vết.